# NB39: MASTER Panel — Reversed-Distribution Eğitim Deneyi

**Hipotez:** Modeli final test dağılımına (%80 benign / %20 pathogenic) yakın bir eğitim setiyle eğitmek,
post-hoc düzeltmelere (threshold tuning, calibrate-then-shift) göre daha etkili olabilir.

**Kanıt:** KANSER panelinde P9_REVERSE_6040 F1_8020'yi 0.69→0.73'e taşıdı.

| Senaryo | Eğitim Kompozisyonu | Tahmini n_train | Benign:Patho |
|---|---|---|---|
| S0_baseline | X_train olduğu gibi | ~2345 | ~73:27 |
| S1_REVERSE_6040 | 60% benign / 40% patho | ~1040 | 60:40 |
| S2_REVERSE_8020 | 80% benign / 20% patho | ~780 | 80:20 |
| S3_REVERSE_7030 | 70% benign / 30% patho | ~890 | 70:30 |
| S4_WEIGHTED | Tamamı + class_weight 80:20 simülasyonu | ~2345 | ağırlıklı |

Her senaryoda **4 model**: LightGBM, BalancedBag_XGB, RF, CatBoost (NN hariç — küçük eğitim seti overfit riski).

**Referans:** NB38 V1_calib F1_8020 = 0.6165

In [1]:
# Cell 1: Imports ve Setup
import sys, os, warnings, time, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.getcwd()
    while PROJECT_ROOT != '/' and not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from config import SEED, TEST_SIZE, PROJECT_ROOT, REPORTS_DIR
from src.columns_real import (
    ID_COL, TARGET_COL, NON_FEATURE_COLS,
    get_constant_cols, get_duplicate_col_pairs, get_missing_mask_col_name
)
from src.metrics import optimize_threshold, compute_all_metrics

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
from sklearn.metrics import precision_score, recall_score
from imblearn.ensemble import BalancedBaggingClassifier

import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

from fpdf import FPDF

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v22_master_reversed_distribution')
os.makedirs(RESULTS_DIR, exist_ok=True)

N_BOOTSTRAP = 50
N_FOLDS = 5

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'SEED: {SEED}, TEST_SIZE: {TEST_SIZE}')
print(f'Results: {RESULTS_DIR}')

PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model
SEED: 42, TEST_SIZE: 0.2
Results: /Users/tefe/teknofest_model/teknofest_model/results/v22_master_reversed_distribution


In [2]:
# Cell 2: Veri Yükleme ve Sütun Temizliği
MASTER_CSV = os.path.join(PROJECT_ROOT, 'data/real_data/YARISMA_TRAIN_MASTER.csv')
df = pd.read_csv(MASTER_CSV)

print(f'Veri yüklendi: {df.shape}')
print(f'Label dağılımı:\n{df[TARGET_COL].value_counts()}')

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

Veri yüklendi: (2931, 353)
Label dağılımı:
Label
1    2149
0     782
Name: count, dtype: int64


In [3]:
# Cell 3: Stratified Split (NB38 ile birebir aynı SEED=42)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Sabit ve özdeş sütun temizliği (train üzerinde tespit)
const_cols = get_constant_cols(X_train_raw)
dup_col_pairs = get_duplicate_col_pairs(X_train_raw)
dup_cols = set(c2 for c1, c2 in dup_col_pairs)

drop_cols = list(set(const_cols + list(dup_cols)))
if 'CAT_6' in X_train_raw.columns and 'CAT_6' not in drop_cols:
    drop_cols.append('CAT_6')

feature_cols = [c for c in feature_cols if c not in drop_cols]
X_train_raw = X_train_raw[feature_cols].copy()
X_test_raw = X_test_raw[feature_cols].copy()

print(f'Train: {X_train_raw.shape} (pos={y_train.sum()}, neg={(y_train==0).sum()})')
print(f'Test:  {X_test_raw.shape} (pos={y_test.sum()}, neg={(y_test==0).sum()})')
print(f'Sabit={len(const_cols)}, Özdeş çift={len(dup_col_pairs)}, Toplam drop={len(drop_cols)}, Kalan={len(feature_cols)}')

Train: (2344, 287) (pos=1719, neg=625)
Test:  (587, 287) (pos=430, neg=157)
Sabit=57, Özdeş çift=583, Toplam drop=64, Kalan=287


In [4]:
# Cell 4: Reversed-Distribution Yardımcı Fonksiyonları

def make_reversed_subset(X_train, y_train, benign_frac, seed=SEED):
    """X_train'den belirli benign/patho oranında alt-küme seç."""
    idx_ben = np.where(y_train.values == 0)[0]
    idx_pat = np.where(y_train.values == 1)[0]
    rng = np.random.RandomState(seed)

    if benign_frac >= 0.80:
        # S2: çok agresif — benign'den de kıs
        n_pat = min(len(idx_pat), int(len(idx_ben) * (1 - benign_frac) / benign_frac))
        n_ben = int(n_pat * benign_frac / (1 - benign_frac))
        ben_sample = rng.choice(idx_ben, size=n_ben, replace=False)
        pat_sample = rng.choice(idx_pat, size=n_pat, replace=False)
    else:
        # S1, S3: tüm benign'i al, pathogenic'i kıs
        n_ben = len(idx_ben)
        n_pat = int(n_ben * (1 - benign_frac) / benign_frac)
        ben_sample = idx_ben
        pat_sample = rng.choice(idx_pat, size=min(n_pat, len(idx_pat)), replace=False)

    idx = np.concatenate([ben_sample, pat_sample])
    rng.shuffle(idx)
    return X_train.iloc[idx].copy(), y_train.iloc[idx].copy()


def compute_sample_weights(y_train, target_benign_frac=0.80):
    """Her örneğe final dağılımına göre ağırlık ata."""
    cur_ben_frac = (y_train == 0).mean()
    cur_pat_frac = (y_train == 1).mean()
    w_ben = target_benign_frac / cur_ben_frac
    w_pat = (1 - target_benign_frac) / cur_pat_frac
    weights = np.where(y_train == 0, w_ben, w_pat)
    return weights


# Test
for name, frac in [('S1_6040', 0.60), ('S2_8020', 0.80), ('S3_7030', 0.70)]:
    Xs, ys = make_reversed_subset(X_train_raw, y_train, frac)
    actual_ben = (ys == 0).mean()
    print(f'{name}: n={len(ys)}, benign_frac={actual_ben:.3f} (hedef={frac}), pos={ys.sum()}, neg={(ys==0).sum()}')

sw = compute_sample_weights(y_train)
print(f'\nS4 weights: w_ben={sw[y_train.values==0][0]:.4f}, w_pat={sw[y_train.values==1][0]:.4f}')

S1_6040: n=1041, benign_frac=0.600 (hedef=0.6), pos=416, neg=625
S2_8020: n=780, benign_frac=0.800 (hedef=0.8), pos=156, neg=624
S3_7030: n=892, benign_frac=0.701 (hedef=0.7), pos=267, neg=625

S4 weights: w_ben=3.0003, w_pat=0.2727


In [5]:
# Cell 5: Preprocessing Pipeline (senaryo-bazlı)

def preprocess_scenario(X_tr, X_te, feature_cols):
    """M3 missing + LabelEncode + medyan imputation. Train'e fit, test'e transform."""
    X_tr = X_tr.copy()
    X_te = X_te.copy()

    # M3: is_missing flags (>%50 missing sütunlar)
    missing_threshold = 0.50
    for col in X_tr.columns:
        miss_ratio = X_tr[col].isnull().sum() / len(X_tr)
        if miss_ratio > missing_threshold:
            mask_col = get_missing_mask_col_name(col)
            X_tr[mask_col] = X_tr[col].isnull().astype(int)
            X_te[mask_col] = X_te[col].isnull().astype(int)

    # Medyan imputation (numeric)
    numeric_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    imputer = SimpleImputer(strategy='median')
    if numeric_cols:
        X_tr[numeric_cols] = imputer.fit_transform(X_tr[numeric_cols])
        X_te[numeric_cols] = imputer.transform(X_te[numeric_cols])

    # LabelEncode kategorikler
    cat_like = [c for c in X_tr.columns if X_tr[c].dtype == object or str(X_tr[c].dtype) == 'category']
    for col in cat_like:
        le = LabelEncoder()
        tr_vals = X_tr[col].astype('object').where(X_tr[col].notna(), '__NA__').astype(str)
        fit_vals = pd.concat([tr_vals, pd.Series(['__NA__'])], ignore_index=True)
        le.fit(fit_vals)
        X_tr[col] = le.transform(tr_vals)
        known = set(le.classes_)
        te_vals = X_te[col].astype('object').where(X_te[col].notna(), '__NA__').astype(str)
        te_vals = te_vals.map(lambda v, k=known: v if v in k else '__NA__')
        X_te[col] = le.transform(te_vals)

    # Kalan NaN'ları sıfırla
    for col in X_tr.columns:
        tr_num = pd.to_numeric(X_tr[col], errors='coerce')
        te_num = pd.to_numeric(X_te[col], errors='coerce')
        fill = tr_num.median()
        if pd.isna(fill):
            fill = 0.0
        X_tr[col] = tr_num.fillna(fill).astype(float)
        X_te[col] = te_num.fillna(fill).astype(float)

    return X_tr, X_te, imputer


print('Preprocessing pipeline hazır.')

Preprocessing pipeline hazır.


In [6]:
# Cell 6: Değerlendirme Fonksiyonları (NB38 birebir)

def optimize_threshold_8020(y_true, y_prob, n_bootstrap=N_BOOTSTRAP, target_pos_rate=0.20):
    best_thrs_f1, best_thrs_mcc = [], []
    rng = np.random.RandomState(SEED)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        best_f1, best_t_f1 = 0, 0.5
        best_mcc, best_t_mcc = -1, 0.5
        for thr in np.arange(0.10, 0.90, 0.01):
            preds = (p_sel >= thr).astype(int)
            f1 = f1_score(y_sel, preds, zero_division=0)
            mcc = matthews_corrcoef(y_sel, preds)
            if f1 > best_f1:
                best_f1, best_t_f1 = f1, thr
            if mcc > best_mcc:
                best_mcc, best_t_mcc = mcc, thr
        best_thrs_f1.append(best_t_f1)
        best_thrs_mcc.append(best_t_mcc)
    return np.median(best_thrs_f1), np.median(best_thrs_mcc)


def bootstrap_8020_eval(y_true, y_prob, threshold, n_bootstrap=N_BOOTSTRAP, target_pos_rate=0.20):
    rng = np.random.RandomState(SEED + 1)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    f1s, mccs, precs, recs = [], [], [], []
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        preds = (p_sel >= threshold).astype(int)
        f1s.append(f1_score(y_sel, preds, zero_division=0))
        mccs.append(matthews_corrcoef(y_sel, preds))
        precs.append(precision_score(y_sel, preds, zero_division=0))
        recs.append(recall_score(y_sel, preds, zero_division=0))
    return {
        'f1_mean': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
        'f1_ci_lo': float(np.percentile(f1s, 2.5)), 'f1_ci_hi': float(np.percentile(f1s, 97.5)),
        'mcc_mean': float(np.mean(mccs)), 'mcc_std': float(np.std(mccs)),
        'prec_mean': float(np.mean(precs)), 'rec_mean': float(np.mean(recs)),
    }


def full_eval(name, y_true, y_prob, y_train_for_thr=None, proba_train_for_thr=None, verbose=True):
    y_thr = y_train_for_thr if y_train_for_thr is not None else y_true
    p_thr = proba_train_for_thr if proba_train_for_thr is not None else y_prob
    thr_raw, f1_raw = optimize_threshold(y_thr, p_thr)
    thr_8020, thr_mcc = optimize_threshold_8020(y_thr, p_thr)
    y_pred = (y_prob >= thr_8020).astype(int)
    m = compute_all_metrics(y_true, y_pred, y_prob)
    bs = bootstrap_8020_eval(y_true, y_prob, thr_8020)
    if verbose:
        print(f'  [{name}] thr_8020={thr_8020:.3f} | F1_5050={m["f1"]:.4f} | '
              f'F1_8020={bs["f1_mean"]:.4f} [{bs["f1_ci_lo"]:.3f}-{bs["f1_ci_hi"]:.3f}]')
    return {
        'name': name, 'thr_raw': thr_raw, 'thr_8020': thr_8020, 'thr_mcc': thr_mcc,
        'f1_5050': m['f1'], 'mcc_5050': m['mcc'], 'auc_roc': m['auc_roc'], 'auc_pr': m['auc_pr'],
        'prec_5050': m['precision'], 'rec_5050': m['recall'],
        'f1_8020': bs['f1_mean'], 'f1_8020_std': bs['f1_std'],
        'f1_8020_ci_lo': bs['f1_ci_lo'], 'f1_8020_ci_hi': bs['f1_ci_hi'],
        'mcc_8020': bs['mcc_mean'], 'prec_8020': bs['prec_mean'], 'rec_8020': bs['rec_mean'],
    }


print('Değerlendirme fonksiyonları hazır.')

Değerlendirme fonksiyonları hazır.


In [7]:
# Cell 7: Model Eğitim Pipeline'ı (senaryo-agnostik)

def train_scenario_models(scenario_name, X_tr, y_tr, X_te, y_te, n_folds=N_FOLDS,
                          sample_weights=None):
    """Bir senaryo için 4 modeli eğit ve değerlendir."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    fold_indices = list(skf.split(X_tr, y_tr))
    results = OrderedDict()

    def oof_train(name, fit_fn, X_train_v, X_test_v):
        oof_train_proba = np.zeros(len(X_train_v))
        oof_test_proba = np.zeros(len(X_test_v))
        train_f1s, val_f1s = [], []
        for fold_i, (tr_idx, val_idx) in enumerate(fold_indices):
            Xf_tr = X_train_v.iloc[tr_idx]
            Xf_val = X_train_v.iloc[val_idx]
            yf_tr = y_tr.iloc[tr_idx]
            yf_val = y_tr.iloc[val_idx]
            sw_fold = sample_weights[tr_idx] if sample_weights is not None else None
            model = fit_fn(Xf_tr, yf_tr, Xf_val, yf_val, sw_fold)
            val_prob = model.predict_proba(Xf_val)[:, 1]
            test_prob = model.predict_proba(X_test_v)[:, 1]
            oof_train_proba[val_idx] = val_prob
            oof_test_proba += test_prob / n_folds
            tr_prob = model.predict_proba(Xf_tr)[:, 1]
            train_f1s.append(f1_score(yf_tr, (tr_prob >= 0.5).astype(int), zero_division=0))
            val_f1s.append(f1_score(yf_val, (val_prob >= 0.5).astype(int), zero_division=0))
        avg_gap = np.mean(train_f1s) - np.mean(val_f1s)
        print(f'  [{scenario_name}/{name}] train_F1={np.mean(train_f1s):.4f} val_F1={np.mean(val_f1s):.4f} gap={avg_gap:+.4f}')
        return oof_train_proba, oof_test_proba, avg_gap

    # --- LightGBM ---
    def fit_lgbm(Xtr, ytr, Xval, yval, sw):
        m = lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=8,
            colsample_bytree=0.8, subsample=0.8, reg_alpha=0.1, reg_lambda=1.0,
            class_weight='balanced', random_state=SEED, verbose=-1
        )
        m.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(20, verbose=False)])
        return m

    # --- BalancedBag_XGB ---
    def fit_balbag(Xtr, ytr, Xval, yval, sw):
        base_est = lgb.LGBMClassifier(
            n_estimators=100, learning_rate=0.1, num_leaves=31,
            random_state=SEED, verbose=-1
        )
        m = BalancedBaggingClassifier(
            estimator=base_est, n_estimators=15,
            sampling_strategy='auto', replacement=False,
            random_state=SEED, n_jobs=-1
        )
        m.fit(Xtr, ytr)
        return m

    # --- RandomForest ---
    def fit_rf(Xtr, ytr, Xval, yval, sw):
        m = RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=3,
            class_weight='balanced', random_state=SEED, n_jobs=-1
        )
        m.fit(Xtr, ytr, sample_weight=sw)
        return m

    # --- CatBoost ---
    def fit_catboost(Xtr, ytr, Xval, yval, sw):
        m = CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6,
            auto_class_weights='Balanced',
            random_seed=SEED, verbose=0
        )
        m.fit(Xtr, ytr, sample_weight=sw, eval_set=(Xval, yval), early_stopping_rounds=20)
        return m

    model_fns = [('lgbm', fit_lgbm), ('balbag', fit_balbag), ('rf', fit_rf), ('catboost', fit_catboost)]

    for model_name, fit_fn in model_fns:
        oof_tr, oof_te, gap = oof_train(model_name, fit_fn, X_tr, X_te)
        res = full_eval(f'{scenario_name}/{model_name}', y_te, oof_te,
                        y_train_for_thr=y_tr, proba_train_for_thr=oof_tr)
        # Train metrikleri
        res_tr = full_eval(f'{scenario_name}/{model_name}_TRAIN', y_tr, oof_tr, verbose=False)
        res['train_f1'] = res_tr['f1_5050']
        res['train_gap'] = gap
        results[model_name] = res

    return results


print('Model eğitim pipeline hazır.')

Model eğitim pipeline hazır.


In [8]:
# Cell 8: S0_BASELINE — Orijinal dağılım (kontrol)
print('=' * 70)
print('S0_BASELINE: Orijinal dağılım (~73:27 patho:benign)')
print('=' * 70)

X_tr_s0, X_te_s0, imp_s0 = preprocess_scenario(X_train_raw, X_test_raw, feature_cols)
results_s0 = train_scenario_models('S0_baseline', X_tr_s0, y_train, X_te_s0, y_test)

print(f'\nS0 tamamlandı. n_train={len(y_train)}')
for m, r in results_s0.items():
    print(f'  {m}: F1_8020={r["f1_8020"]:.4f}')

S0_BASELINE: Orijinal dağılım (~73:27 patho:benign)
  [S0_baseline/lgbm] train_F1=0.9631 val_F1=0.8632 gap=+0.0999
  [S0_baseline/lgbm] thr_8020=0.750 | F1_5050=0.7794 | F1_8020=0.5511 [0.453-0.640]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [S0_baseline/balbag] train_F1=0.9411 val_F1=0.8443 gap=+0.0967
  [S0_baseline/balbag] thr_8020=0.700 | F1_5050=0.7963 | F1_8020=0.5747 [0.485-0.646]
  [S0_baseline/rf] train_F1=0.9588 val_F1=0.8646 gap=+0.0941
  [S0_baseline/rf] thr_8020=0.710 | F1_5050=0.7926 | F1_8020=0.6087 [0.497-0.696]
  [S0_baseline/catboost] train_F1=0.9321 val_F1=0.8568 gap=+0.0753
  [S0_baseline/catboost] thr_8020=0.630 | F1_5050=0.7990 | F1_8020=0.5810 [0.491-0.663]

S0 tamamlandı. n_train=2344
  lgbm: F1_8020=0.5511
  balbag: F1_8020=0.5747
  rf: F1_8020=0.6087
  catboost: F1_8020=0.5810


In [9]:
# Cell 9: S1_REVERSE_6040 — %60 benign / %40 pathogenic
print('=' * 70)
print('S1_REVERSE_6040: %60 benign / %40 pathogenic')
print('=' * 70)

X_tr_s1_raw, y_tr_s1 = make_reversed_subset(X_train_raw, y_train, benign_frac=0.60)
X_tr_s1, X_te_s1, imp_s1 = preprocess_scenario(X_tr_s1_raw, X_test_raw, feature_cols)
results_s1 = train_scenario_models('S1_6040', X_tr_s1, y_tr_s1, X_te_s1, y_test)

print(f'\nS1 tamamlandı. n_train={len(y_tr_s1)}, benign_frac={(y_tr_s1==0).mean():.3f}')
for m, r in results_s1.items():
    print(f'  {m}: F1_8020={r["f1_8020"]:.4f}')

S1_REVERSE_6040: %60 benign / %40 pathogenic
  [S1_6040/lgbm] train_F1=0.9230 val_F1=0.7029 gap=+0.2200
  [S1_6040/lgbm] thr_8020=0.545 | F1_5050=0.7758 | F1_8020=0.5804 [0.491-0.656]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [S1_6040/balbag] train_F1=0.9440 val_F1=0.6910 gap=+0.2530
  [S1_6040/balbag] thr_8020=0.550 | F1_5050=0.8021 | F1_8020=0.6379 [0.519-0.716]
  [S1_6040/rf] train_F1=0.9140 val_F1=0.6551 gap=+0.2590
  [S1_6040/rf] thr_8020=0.480 | F1_5050=0.7877 | F1_8020=0.6065 [0.489-0.680]
  [S1_6040/catboost] train_F1=0.8892 val_F1=0.6910 gap=+0.1983
  [S1_6040/catboost] thr_8020=0.550 | F1_5050=0.7740 | F1_8020=0.5963 [0.480-0.665]

S1 tamamlandı. n_train=1041, benign_frac=0.600
  lgbm: F1_8020=0.5804
  balbag: F1_8020=0.6379
  rf: F1_8020=0.6065
  catboost: F1_8020=0.5963


In [10]:
# Cell 10: S2_REVERSE_8020 — %80 benign / %20 pathogenic (agresif)
print('=' * 70)
print('S2_REVERSE_8020: %80 benign / %20 pathogenic (agresif)')
print('=' * 70)

X_tr_s2_raw, y_tr_s2 = make_reversed_subset(X_train_raw, y_train, benign_frac=0.80)
X_tr_s2, X_te_s2, imp_s2 = preprocess_scenario(X_tr_s2_raw, X_test_raw, feature_cols)
results_s2 = train_scenario_models('S2_8020', X_tr_s2, y_tr_s2, X_te_s2, y_test,
                                    n_folds=5)

print(f'\nS2 tamamlandı. n_train={len(y_tr_s2)}, benign_frac={(y_tr_s2==0).mean():.3f}')
for m, r in results_s2.items():
    print(f'  {m}: F1_8020={r["f1_8020"]:.4f}')

S2_REVERSE_8020: %80 benign / %20 pathogenic (agresif)
  [S2_8020/lgbm] train_F1=0.8939 val_F1=0.4628 gap=+0.4311
  [S2_8020/lgbm] thr_8020=0.250 | F1_5050=0.8090 | F1_8020=0.5146 [0.448-0.600]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [S2_8020/balbag] train_F1=0.7561 val_F1=0.4703 gap=+0.2857
  [S2_8020/balbag] thr_8020=0.520 | F1_5050=0.7157 | F1_8020=0.5486 [0.424-0.643]
  [S2_8020/rf] train_F1=0.8497 val_F1=0.3674 gap=+0.4823
  [S2_8020/rf] thr_8020=0.325 | F1_5050=0.7503 | F1_8020=0.4876 [0.371-0.587]
  [S2_8020/catboost] train_F1=0.6872 val_F1=0.4338 gap=+0.2533
  [S2_8020/catboost] thr_8020=0.425 | F1_5050=0.8255 | F1_8020=0.4915 [0.424-0.552]

S2 tamamlandı. n_train=780, benign_frac=0.800
  lgbm: F1_8020=0.5146
  balbag: F1_8020=0.5486
  rf: F1_8020=0.4876
  catboost: F1_8020=0.4915


In [11]:
# Cell 11: S3_REVERSE_7030 — %70 benign / %30 pathogenic
print('=' * 70)
print('S3_REVERSE_7030: %70 benign / %30 pathogenic')
print('=' * 70)

X_tr_s3_raw, y_tr_s3 = make_reversed_subset(X_train_raw, y_train, benign_frac=0.70)
X_tr_s3, X_te_s3, imp_s3 = preprocess_scenario(X_tr_s3_raw, X_test_raw, feature_cols)
results_s3 = train_scenario_models('S3_7030', X_tr_s3, y_tr_s3, X_te_s3, y_test)

print(f'\nS3 tamamlandı. n_train={len(y_tr_s3)}, benign_frac={(y_tr_s3==0).mean():.3f}')
for m, r in results_s3.items():
    print(f'  {m}: F1_8020={r["f1_8020"]:.4f}')

S3_REVERSE_7030: %70 benign / %30 pathogenic
  [S3_7030/lgbm] train_F1=0.9119 val_F1=0.6280 gap=+0.2839
  [S3_7030/lgbm] thr_8020=0.490 | F1_5050=0.7568 | F1_8020=0.5700 [0.446-0.657]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [S3_7030/balbag] train_F1=0.8824 val_F1=0.6292 gap=+0.2532
  [S3_7030/balbag] thr_8020=0.585 | F1_5050=0.7503 | F1_8020=0.5738 [0.439-0.652]
  [S3_7030/rf] train_F1=0.9046 val_F1=0.5766 gap=+0.3281
  [S3_7030/rf] thr_8020=0.420 | F1_5050=0.7694 | F1_8020=0.5621 [0.429-0.656]
  [S3_7030/catboost] train_F1=0.8623 val_F1=0.6360 gap=+0.2263
  [S3_7030/catboost] thr_8020=0.520 | F1_5050=0.7497 | F1_8020=0.5823 [0.465-0.671]

S3 tamamlandı. n_train=892, benign_frac=0.701
  lgbm: F1_8020=0.5700
  balbag: F1_8020=0.5738
  rf: F1_8020=0.5621
  catboost: F1_8020=0.5823


In [12]:
# Cell 12: S4_WEIGHTED — Tüm veri + sample_weight ile %80/20 simülasyonu
print('=' * 70)
print('S4_WEIGHTED: Tüm veri + sample_weight (%80/20 simülasyonu)')
print('=' * 70)

X_tr_s4, X_te_s4, imp_s4 = preprocess_scenario(X_train_raw, X_test_raw, feature_cols)
sw_s4 = compute_sample_weights(y_train, target_benign_frac=0.80)
results_s4 = train_scenario_models('S4_weighted', X_tr_s4, y_train, X_te_s4, y_test,
                                    sample_weights=sw_s4)

print(f'\nS4 tamamlandı. n_train={len(y_train)}, w_ben={sw_s4[y_train.values==0][0]:.3f}, w_pat={sw_s4[y_train.values==1][0]:.3f}')
for m, r in results_s4.items():
    print(f'  {m}: F1_8020={r["f1_8020"]:.4f}')

S4_WEIGHTED: Tüm veri + sample_weight (%80/20 simülasyonu)
  [S4_weighted/lgbm] train_F1=0.9613 val_F1=0.8144 gap=+0.1469
  [S4_weighted/lgbm] thr_8020=0.700 | F1_5050=0.7514 | F1_8020=0.5917 [0.483-0.673]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [S4_weighted/balbag] train_F1=0.9411 val_F1=0.8443 gap=+0.0967
  [S4_weighted/balbag] thr_8020=0.700 | F1_5050=0.7963 | F1_8020=0.5747 [0.485-0.646]
  [S4_weighted/rf] train_F1=0.9040 val_F1=0.7974 gap=+0.1066
  [S4_weighted/rf] thr_8020=0.610 | F1_5050=0.7056 | F1_8020=0.4708 [0.383-0.560]
  [S4_weighted/catboost] train_F1=0.9614 val_F1=0.8598 gap=+0.1017
  [S4_weighted/catboost] thr_8020=0.720 | F1_5050=0.8098 | F1_8020=0.5911 [0.508-0.665]

S4 tamamlandı. n_train=2344, w_ben=3.000, w_pat=0.273
  lgbm: F1_8020=0.5917
  balbag: F1_8020=0.5747
  rf: F1_8020=0.4708
  catboost: F1_8020=0.5911


In [13]:
# Cell 13: Büyük Karşılaştırma Tablosu (5 senaryo × 4 model = 20 satır)
print('=' * 90)
print('KAPSAMLI KARŞILAŞTIRMA TABLOSU (5 senaryo × 4 model)')
print('=' * 90)

all_scenario_results = OrderedDict()
scenario_meta = {}
for scn_name, scn_results, n_tr in [
    ('S0_baseline', results_s0, len(y_train)),
    ('S1_6040', results_s1, len(y_tr_s1)),
    ('S2_8020', results_s2, len(y_tr_s2)),
    ('S3_7030', results_s3, len(y_tr_s3)),
    ('S4_weighted', results_s4, len(y_train)),
]:
    scenario_meta[scn_name] = n_tr
    for model_name, res in scn_results.items():
        key = f'{scn_name}/{model_name}'
        all_scenario_results[key] = res
        all_scenario_results[key]['scenario'] = scn_name
        all_scenario_results[key]['model'] = model_name
        all_scenario_results[key]['n_train'] = n_tr

# NB38 referans
all_scenario_results['NB38_V1_calib'] = {
    'name': 'NB38_V1_calib', 'scenario': 'REF', 'model': 'hetstack',
    'f1_8020': 0.6165, 'f1_8020_ci_lo': None, 'f1_8020_ci_hi': None,
    'f1_5050': 0.6165, 'mcc_8020': None, 'prec_8020': None, 'rec_8020': None,
    'train_f1': None, 'train_gap': None, 'auc_roc': None, 'n_train': 2345,
    'thr_raw': None, 'thr_8020': None, 'thr_mcc': None,
}

rows = []
for key, res in all_scenario_results.items():
    ci = ''
    if res.get('f1_8020_ci_lo') is not None:
        ci = f"[{res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]"
    rows.append({
        'Model': key,
        'Scenario': res.get('scenario', ''),
        'n_train': res.get('n_train', ''),
        'F1_8020': round(res.get('f1_8020', 0), 4),
        'CI_95': ci,
        'MCC_8020': round(res.get('mcc_8020', 0) or 0, 4),
        'Prec_8020': round(res.get('prec_8020', 0) or 0, 4),
        'Rec_8020': round(res.get('rec_8020', 0) or 0, 4),
        'F1_5050': round(res.get('f1_5050', 0) or 0, 4),
        'AUC': round(res.get('auc_roc', 0) or 0, 4),
        'Gap': round(res.get('train_gap', 0) or 0, 4),
        'Thr_raw': round(res.get('thr_raw', 0) or 0, 3),
        'Thr_8020': round(res.get('thr_8020', 0) or 0, 3),
    })

comparison_df = pd.DataFrame(rows).sort_values('F1_8020', ascending=False)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(os.path.join(RESULTS_DIR, 'reversed_distribution_results.csv'), index=False)

best_row = comparison_df.iloc[0]
print(f'\n→ EN İYİ: {best_row["Model"]} | F1_8020={best_row["F1_8020"]:.4f} {best_row["CI_95"]}')
print(f'→ NB38 referans: 0.6165')
print(f'→ İyileştirme: {best_row["F1_8020"] - 0.6165:+.4f}')

KAPSAMLI KARŞILAŞTIRMA TABLOSU (5 senaryo × 4 model)
               Model    Scenario  n_train  F1_8020         CI_95  MCC_8020  Prec_8020  Rec_8020  F1_5050    AUC    Gap  Thr_raw  Thr_8020
      S1_6040/balbag     S1_6040     1041   0.6379 [0.519-0.716]    0.5423     0.5829    0.7082   0.8021 0.8317 0.2530     0.38     0.550
       NB38_V1_calib         REF     2345   0.6165                  0.0000     0.0000    0.0000   0.6165 0.0000 0.0000     0.00     0.000
      S0_baseline/rf S0_baseline     2344   0.6087 [0.497-0.696]    0.5042     0.5453    0.6933   0.7926 0.8389 0.0941     0.38     0.710
          S1_6040/rf     S1_6040     1041   0.6065 [0.489-0.680]    0.5013     0.5448    0.6882   0.7877 0.8261 0.2590     0.37     0.480
    S1_6040/catboost     S1_6040     1041   0.5963 [0.480-0.665]    0.4889     0.5444    0.6636   0.7740 0.8374 0.1983     0.48     0.550
    S4_weighted/lgbm S4_weighted     2344   0.5917 [0.483-0.673]    0.4849     0.5523    0.6421   0.7514 0.8354 0.1469 

In [14]:
# Cell 14: Görselleştirmeler

# --- Fig 1: Senaryo × Model F1_8020 Heatmap ---
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Heatmap data
scenarios = ['S0_baseline', 'S1_6040', 'S2_8020', 'S3_7030', 'S4_weighted']
models = ['lgbm', 'balbag', 'rf', 'catboost']
heatmap_data = pd.DataFrame(index=scenarios, columns=models, dtype=float)
for scn in scenarios:
    scn_res = [r for k, r in all_scenario_results.items() if r.get('scenario') == scn]
    for r in scn_res:
        if r.get('model') in models:
            heatmap_data.loc[scn, r['model']] = r['f1_8020']

ax = axes[0, 0]
sns.heatmap(heatmap_data.astype(float), annot=True, fmt='.4f', cmap='RdYlGn',
            ax=ax, vmin=0.50, vmax=0.70, linewidths=0.5)
ax.set_title('F1_8020: Senaryo × Model')
ax.set_ylabel('Senaryo')

# --- Fig 2: Precision Karşılaştırma ---
ax = axes[0, 1]
prec_data = pd.DataFrame(index=scenarios, columns=models, dtype=float)
for scn in scenarios:
    scn_res = [r for k, r in all_scenario_results.items() if r.get('scenario') == scn]
    for r in scn_res:
        if r.get('model') in models:
            prec_data.loc[scn, r['model']] = r.get('prec_8020', 0)

prec_data.astype(float).plot(kind='bar', ax=ax)
ax.set_title('Precision_8020: Senaryo × Model')
ax.set_ylabel('Precision')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(fontsize=7)
ax.axhline(y=0.55, color='red', linestyle='--', alpha=0.5, label='NB38 best')

# --- Fig 3: n_train vs F1_8020 ---
ax = axes[0, 2]
for model_name in models:
    xs, ys = [], []
    for scn in scenarios:
        key = f'{scn}/{model_name}'
        if key in all_scenario_results:
            xs.append(all_scenario_results[key].get('n_train', 0))
            ys.append(all_scenario_results[key]['f1_8020'])
    ax.plot(xs, ys, 'o-', label=model_name, markersize=6)
ax.set_xlabel('n_train')
ax.set_ylabel('F1_8020')
ax.set_title('Veri Boyutu vs F1_8020')
ax.legend(fontsize=7)
ax.axhline(y=0.6165, color='red', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

# --- Fig 4: En iyi 4 modelin confusion matrix ---
top4 = comparison_df[comparison_df['Scenario'] != 'REF'].head(4)
for i, (_, row) in enumerate(top4.iterrows()):
    ax = axes[1, i] if i < 3 else None
    if ax is None:
        break
    key = row['Model']
    res = all_scenario_results.get(key)
    if res is None:
        continue
    # Confusion matrix hesapla
    # Not: proba saklı değil, sadece metrikleri göster
    ax.text(0.5, 0.5, f"{key}\nF1_8020={row['F1_8020']:.4f}\nPrec={row['Prec_8020']:.3f}\nRec={row['Rec_8020']:.3f}",
            ha='center', va='center', fontsize=9, transform=ax.transAxes)
    ax.set_title(f'Top {i+1}: {key.split("/")[-1]}', fontsize=9)
    ax.axis('off')

# --- Fig 5: Train-Test Gap ---
if len(axes[1]) > 2:
    # Fill remaining
    pass

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'all_experiments.png'), dpi=150, bbox_inches='tight')
print(f'Görselleştirme kaydedildi: {RESULTS_DIR}/all_experiments.png')
plt.close()

# --- Fig 6: Threshold Convergence (f1_raw vs f1_8020) ---
fig2, ax2 = plt.subplots(1, 1, figsize=(10, 6))
for model_name in models:
    thr_raws, thr_8020s, scn_labels = [], [], []
    for scn in scenarios:
        key = f'{scn}/{model_name}'
        if key in all_scenario_results:
            r = all_scenario_results[key]
            if r.get('thr_raw') is not None:
                thr_raws.append(r['thr_raw'])
                thr_8020s.append(r['thr_8020'])
                scn_labels.append(scn)
    if thr_raws:
        diffs = [abs(a - b) for a, b in zip(thr_raws, thr_8020s)]
        ax2.plot(scn_labels, diffs, 'o-', label=model_name)

ax2.set_xlabel('Senaryo')
ax2.set_ylabel('|thr_raw - thr_8020|')
ax2.set_title('Threshold Yakınsama: Eğitim dağılımı test dağılımına yaklaştıkça fark azalmalı')
ax2.legend()
ax2.grid(True, alpha=0.3)
fig2.savefig(os.path.join(RESULTS_DIR, 'threshold_convergence.png'), dpi=150, bbox_inches='tight')
print(f'Threshold yakınsama grafiği kaydedildi.')
plt.close()

Görselleştirme kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v22_master_reversed_distribution/all_experiments.png
Threshold yakınsama grafiği kaydedildi.


In [18]:
# Cell 15: Kapsamlı PDF Rapor

class NB39Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB39: MASTER - Reversed-Distribution Egitim Deneyi', 0, 1, 'C')
        self.set_font('Helvetica', '', 9)
        self.cell(0, 5, f'SEED={SEED} | TEST_SIZE={TEST_SIZE} | 5 senaryo x 4 model', 0, 1, 'C')
        self.ln(3)

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f'  {title}', 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body_text(self, text):
        self.set_font('Helvetica', '', 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        self.set_font('Helvetica', 'B', 7)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, 'C', fill=True)
        self.ln()
        self.set_font('Helvetica', '', 6.5)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            if j % 2 == 0:
                self.set_fill_color(236, 240, 241)
            else:
                self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, 'C', fill=True)
            self.ln()
        self.ln(3)


pdf = NB39Report()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# --- 0. Motivasyon ---
pdf.section_title('0. Motivasyon ve Hipotez')
pdf.body_text(
    'NB38 en iyi sonuc: V1_calib F1_8020=0.6165 (heterojen stacking + isotonic kalibrasyon).\n'
    'Calibrate-then-shift MASTER da ek kazanc vermedi. Precision=0.55 darbogazda.\n\n'
    'Hipotez: Modeli final test dagilimina (%80 benign / %20 pathogenic) yakin bir egitim setiyle\n'
    'egitmek, post-hoc duzeltmelere gore daha etkili olabilir.\n'
    'Kanit: KANSER panelinde P9_REVERSE_6040 F1_8020 yi 0.69->0.73 e tasidi.'
)

# --- 1. Deney Tasarımı ---
pdf.section_title('1. Deney Tasarimi')
scn_headers = ['Senaryo', 'Kompozisyon', 'n_train', 'Benign Frac']
scn_rows = []
scn_info = [
    ('S0_baseline', 'Orijinal (~73:27 patho:ben)', len(y_train), f'{(y_train==0).mean():.3f}'),
    ('S1_6040', '%60 benign / %40 patho', len(y_tr_s1), f'{(y_tr_s1==0).mean():.3f}'),
    ('S2_8020', '%80 benign / %20 patho', len(y_tr_s2), f'{(y_tr_s2==0).mean():.3f}'),
    ('S3_7030', '%70 benign / %30 patho', len(y_tr_s3), f'{(y_tr_s3==0).mean():.3f}'),
    ('S4_weighted', 'Tum veri + class_weight', len(y_train), 'agirlikli'),
]
for s in scn_info:
    scn_rows.append(list(s))
pdf.add_table(scn_headers, scn_rows, [35, 55, 30, 30])

pdf.body_text(
    f'Veri: MASTER (n={len(df)}, pos={int(y.sum())}, neg={int((y==0).sum())})\n'
    f'Split: %80/20 stratified (SEED={SEED})\n'
    f'Test: {len(y_test)} (NB38 ile birebir ayni)\n'
    f'Feature temizlik: sabit={len(const_cols)}, ozdes={len(dup_col_pairs)}, kalan={len(feature_cols)}\n'
    f'Missing: M3 (is_missing >%50 + medyan) - her senaryo icin AYRI fit\n'
    f'Modeller: LightGBM, BalancedBag, RF, CatBoost (NN haric)'
)

# --- 2. Senaryo Sonuçları ---
for scn_name, scn_results_dict in [
    ('S0_baseline', results_s0), ('S1_6040', results_s1),
    ('S2_8020', results_s2), ('S3_7030', results_s3), ('S4_weighted', results_s4)
]:
    pdf.section_title(f'2. {scn_name} Sonuclari')
    s_headers = ['Model', 'F1_8020', 'CI_95', 'MCC', 'Prec', 'Rec', 'F1_50', 'Gap']
    s_rows = []
    for mname, res in scn_results_dict.items():
        ci = f"[{res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]"
        s_rows.append([
            mname, f"{res['f1_8020']:.4f}", ci,
            f"{res.get('mcc_8020',0):.3f}", f"{res.get('prec_8020',0):.3f}",
            f"{res.get('rec_8020',0):.3f}", f"{res.get('f1_5050',0):.3f}",
            f"{res.get('train_gap',0):+.3f}"
        ])
    pdf.add_table(s_headers, s_rows, [22, 20, 32, 18, 18, 18, 18, 18])

# --- 3. Büyük Karşılaştırma ---
pdf.add_page()
pdf.section_title('3. Buyuk Karsilastirma Tablosu (tum senaryolar)')
comp_headers = ['Model', 'n_train', 'F1_8020', 'CI_95', 'MCC', 'Prec', 'Rec', 'Gap']
comp_rows = []
for _, row in comparison_df.iterrows():
    comp_rows.append([
        str(row['Model'])[:22], str(row.get('n_train', '')),
        f"{row['F1_8020']:.4f}", str(row['CI_95'])[:18],
        f"{row['MCC_8020']:.3f}", f"{row['Prec_8020']:.3f}",
        f"{row['Rec_8020']:.3f}", f"{row['Gap']:+.3f}"
    ])
pdf.add_table(comp_headers, comp_rows, [38, 18, 20, 30, 18, 18, 18, 18])

# --- 4. Hipotez Testi: Threshold Yakınsama ---
pdf.section_title('4. Hipotez: Threshold Yakinsama')
pdf.body_text(
    'Hipotez: S1/S2/S3 te thr_raw ve thr_8020 arasindaki fark kuculmeli,\n'
    'cunku egitim dagilimi zaten %80/20 ye yakin.\n\n'
    'Threshold farklari:'
)
thr_headers = ['Senaryo/Model', 'thr_raw', 'thr_8020', '|Fark|']
thr_rows = []
for key, res in all_scenario_results.items():
    if res.get('thr_raw') is not None and res.get('thr_8020') is not None:
        diff = abs(res['thr_raw'] - res['thr_8020'])
        thr_rows.append([key[:22], f"{res['thr_raw']:.3f}", f"{res['thr_8020']:.3f}", f"{diff:.3f}"])
pdf.add_table(thr_headers, thr_rows, [50, 30, 30, 30])

# --- 5. Precision Analizi ---
pdf.section_title('5. Precision Analizi')
pdf.body_text(
    'NB38 de precision=0.55 darbogazdi. Reversed-distribution ile\n'
    'precision artisi beklenir (model benign i daha iyi ogrenir, FP azalir).\n\n'
    'Senaryo bazinda ortalama precision:'
)
for scn_name, scn_results_dict in [
    ('S0', results_s0), ('S1', results_s1), ('S2', results_s2),
    ('S3', results_s3), ('S4', results_s4)
]:
    avg_prec = np.mean([r.get('prec_8020', 0) for r in scn_results_dict.values()])
    pdf.body_text(f'  {scn_name}: avg precision_8020 = {avg_prec:.4f}')

# --- 6. Sonuç ve Karar ---
pdf.add_page()
pdf.section_title('6. Sonuc ve Karar')
best_name = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1_8020']
improvement = best_f1 - 0.6165

pdf.body_text(
    f'En iyi model: {best_name} (F1_8020={best_f1:.4f})\n'
    f'NB38 referans: V1_calib = 0.6165\n'
    f'Iyilestirme: {improvement:+.4f}\n'
)

if improvement > 0.02:
    pdf.body_text(
        'SONUC: Reversed-distribution egitim MASTER icin CALISTI.\n'
        f'En iyi senaryo/model: {best_name}. Bu yaklasim final model adayi.'
    )
elif improvement > 0.005:
    pdf.body_text(
        'SONUC: Mutevazi iyilestirme. Reversed-distribution marjinal kazanc veriyor.\n'
        'Stacking ile birlestirildiginde daha fazla kazanc gorulebilir.'
    )
elif improvement > -0.005:
    pdf.body_text(
        'SONUC: NB38 ile esit. Reversed-distribution MASTER da ek kazanc vermiyor.\n'
        'Sorun dagilim degil, sinyalin zayifligi. NB38 V1_calib final model.'
    )
else:
    pdf.body_text(
        'SONUC: Reversed-distribution MASTER da performansi DUSURDU.\n'
        'Veri kaybi etkisi baskin. NB38 V1_calib (0.6165) final model olarak sabitle.'
    )

# S4 vs reversed karşılaştırma
best_s4 = max(results_s4.values(), key=lambda r: r['f1_8020'])
best_reversed = None
for scn_res in [results_s1, results_s2, results_s3]:
    for r in scn_res.values():
        if best_reversed is None or r['f1_8020'] > best_reversed['f1_8020']:
            best_reversed = r

if best_s4['f1_8020'] > best_reversed['f1_8020'] + 0.005:
    pdf.body_text(
        f'S4_WEIGHTED ({best_s4["f1_8020"]:.4f}) reversed senaryolarin en iyisini '
        f'({best_reversed["f1_8020"]:.4f}) gecti.\n'
        'Agirlik ile dagilim simulasyonu veri kaybi olmadan ayni etkiyi veriyor — en temiz cozum.'
    )
elif best_reversed['f1_8020'] > best_s4['f1_8020'] + 0.005:
    pdf.body_text(
        f'Reversed ({best_reversed["name"]}: {best_reversed["f1_8020"]:.4f}) '
        f'S4_WEIGHTED ({best_s4["f1_8020"]:.4f}) yi gecti.\n'
        'Gercek resample, agirliklamadan daha etkili - karar sinirini dogrudan kaydiriyor.'
    )

# Görseller ekle
for img_name in ['all_experiments.png', 'threshold_convergence.png']:
    img_path = os.path.join(RESULTS_DIR, img_name)
    if os.path.exists(img_path):
        pdf.add_page()
        pdf.section_title(f'Gorsel: {img_name}')
        pdf.image(img_path, x=10, w=190)

# Kaydet
report_path = os.path.join(REPORTS_DIR, 'NB39_master_reversed_distribution_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB39_master_reversed_distribution_report.pdf


In [19]:
# Cell 16: Final Özet
print('\n' + '=' * 70)
print('NB39 TAMAMLANDI — REVERSED-DISTRIBUTION EĞİTİM DENEYİ')
print('=' * 70)

print(f'\nEn iyi model: {best_row["Model"]}')
print(f'F1_8020: {best_row["F1_8020"]:.4f} {best_row["CI_95"]}')
print(f'NB38 referans: 0.6165')
print(f'İyileştirme: {best_row["F1_8020"] - 0.6165:+.4f}')

print(f'\nSenaryo bazında en iyi F1_8020:')
for scn in scenarios:
    scn_best = comparison_df[comparison_df['Scenario'] == scn].iloc[0] if len(comparison_df[comparison_df['Scenario'] == scn]) > 0 else None
    if scn_best is not None:
        print(f'  {scn}: {scn_best["Model"]} = {scn_best["F1_8020"]:.4f}')

print(f'\nÇıktılar:')
print(f'  CSV: {RESULTS_DIR}/reversed_distribution_results.csv')
print(f'  PNG: {RESULTS_DIR}/all_experiments.png')
print(f'  PNG: {RESULTS_DIR}/threshold_convergence.png')
print(f'  PDF: {REPORTS_DIR}/NB39_master_reversed_distribution_report.pdf')


NB39 TAMAMLANDI — REVERSED-DISTRIBUTION EĞİTİM DENEYİ

En iyi model: S1_6040/balbag
F1_8020: 0.6379 [0.519-0.716]
NB38 referans: 0.6165
İyileştirme: +0.0214

Senaryo bazında en iyi F1_8020:
  S0_baseline: S0_baseline/rf = 0.6087
  S1_6040: S1_6040/balbag = 0.6379
  S2_8020: S2_8020/balbag = 0.5486
  S3_7030: S3_7030/catboost = 0.5823
  S4_weighted: S4_weighted/lgbm = 0.5917

Çıktılar:
  CSV: /Users/tefe/teknofest_model/teknofest_model/results/v22_master_reversed_distribution/reversed_distribution_results.csv
  PNG: /Users/tefe/teknofest_model/teknofest_model/results/v22_master_reversed_distribution/all_experiments.png
  PNG: /Users/tefe/teknofest_model/teknofest_model/results/v22_master_reversed_distribution/threshold_convergence.png
  PDF: /Users/tefe/teknofest_model/teknofest_model/reports/NB39_master_reversed_distribution_report.pdf
